# Visitor Design Pattern 

explained using the Shapes and Exporting example.

#### The Concept

The Visitor Pattern allows you to **add new operations** to existing objects without modifying their classes.

**The Problem**: You have a stable class hierarchy (like `Circle`, `Square`, `Triangle`). You want to add a new feature (e.g., "Export to XML").

- **Without Visitor**: You have to open every single class (`Circle`, `Square`...) and add an `export_xml()` method.
- **With Visitor**: You create a separate "Visitor" class that contains the export logic for all shapes. The shapes just "accept" the visitor.

**Analogy**: A Tax Auditor (Visitor). The auditor visits a Bank, a Factory, and a Shop. The logic for "auditing" is different for each, but the buildings don't need to know how to audit themselves. They just open the door (accept()) for the auditor.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we use **Double Dispatch**.

- The Visitor has methods for every type: `visit_circle`, `visit_square`.
- The Element has an `accept` method that calls back the specific method on the visitor.

This is verbose but type-safe in languages like Java/C++.

#### THE VISITOR INTERFACE

In [1]:
from abc import ABC, abstractmethod

class Visitor(ABC):
    @abstractmethod
    def visit_dot(self, dot): pass
    
    @abstractmethod
    def visit_circle(self, circle): pass
    
    @abstractmethod
    def visit_rectangle(self, rectangle): pass

#### THE ELEMENTS (Shapes)

In [3]:
from dataclasses import dataclass

class Shape(ABC):
    @abstractmethod
    def accept(self, visitor: Visitor):
        pass

@dataclass
class Dot(Shape):
    x: int
    y: int
    
    def accept(self, visitor: Visitor):
        # Double Dispatch: "I am a Dot, so I call visit_dot"
        visitor.visit_dot(self)

@dataclass
class Circle(Shape):
    radius: int
    
    def accept(self, visitor: Visitor):
        visitor.visit_circle(self)

@dataclass
class Rectangle(Shape):
    width: int
    height: int
    
    def accept(self, visitor: Visitor):
        visitor.visit_rectangle(self)

#### CONCRETE VISITOR (New Operation)

In [4]:
class XMLExportVisitor(Visitor):
    def visit_dot(self, dot):
        print(f"  <dot x='{dot.x}' y='{dot.y}' />")

    def visit_circle(self, circle):
        print(f"  <circle r='{circle.radius}' />")

    def visit_rectangle(self, rect):
        print(f"  <rect w='{rect.width}' h='{rect.height}' />")

class AreaCalculatorVisitor(Visitor):
    def visit_dot(self, dot):
        print("Dot area: 0")

    def visit_circle(self, circle):
        area = 3.14 * (circle.radius ** 2)
        print(f"Circle Area: {area}")

    def visit_rectangle(self, rect):
        print(f"Rect Area: {rect.width * rect.height}")

#### CLIENT CODE

In [5]:
def main():
    shapes = [Dot(1, 2), Circle(5), Rectangle(10, 20)]

    print("--- Exporting to XML ---")
    exporter = XMLExportVisitor()
    for shape in shapes:
        shape.accept(exporter)

    print("\n--- Calculating Area ---")
    calc = AreaCalculatorVisitor()
    for shape in shapes:
        shape.accept(calc)

if __name__ == "__main__":
    main()

--- Exporting to XML ---
  <dot x='1' y='2' />
  <circle r='5' />
  <rect w='10' h='20' />

--- Calculating Area ---
Dot area: 0
Circle Area: 78.5
Rect Area: 200


## The Pythonic Way (Dynamic Dispatch)

The Java way is rigid. If you add a `Triangle`, you must update the `Visitor` interface and every concrete visitor. In Python, we can use **Introspection** (`getattr`) to dispatch calls dynamically based on the class name. This means:
- No `accept` **method needed** on the elements (Shapes can remain pure data classes).
- The Visitor handles the logic entirely.

#### THE ELEMENTS (Pure Data)

In [6]:
from dataclasses import dataclass

@dataclass
class Dot:
    x: int
    y: int

@dataclass
class Circle:
    radius: int

@dataclass
class Rectangle:
    width: int
    height: int

#### THE PYTHONIC VISITOR (Dynamic Dispatch)

In [7]:
class ShapeVisitor:
    """
    Generic Visitor that routes calls based on object type.
    """
    def visit(self, shape):
        # 1. Get the class name (e.g., "Circle")
        method_name = f"visit_{type(shape).__name__}"
        
        # 2. Find the method (e.g., self.visit_Circle)
        # If not found, use self.generic_visit
        visitor_method = getattr(self, method_name, self.generic_visit)
        
        # 3. Execute
        return visitor_method(shape)

    def generic_visit(self, shape):
        print(f"No logic defined for {type(shape).__name__}")

#### CONCRETE VISITORS

In [8]:
class XMLExporter(ShapeVisitor):
    # Method names match the Class Names (Case Sensitive usually, or handled in logic)
    
    def visit_Dot(self, dot):
        print(f"  <dot point='{dot.x},{dot.y}' />")

    def visit_Circle(self, circle):
        print(f"  <circle r='{circle.radius}' />")

    def visit_Rectangle(self, r):
        print(f"  <rect dims='{r.width}x{r.height}' />")

#### CLIENT CODE

In [9]:
def main():
    shapes = [Dot(10, 10), Circle(5), Rectangle(4, 8)]

    exporter = XMLExporter()

    print("--- Pythonic Visitor ---")
    for shape in shapes:
        # The Visitor controls the flow completely
        exporter.visit(shape)

if __name__ == "__main__":
    main()

--- Pythonic Visitor ---
  <dot point='10,10' />
  <circle r='5' />
  <rect dims='4x8' />


#### Key Differences

| Feature              | Classic OOP                                                        | Pythonic                                                      |
|----------------------|--------------------------------------------------------------------|----------------------------------------------------------------|
| **Coupling**         | High — elements must know the Visitor interface (`accept` method). | Zero — elements are oblivious; only the visitor knows elements. |
| **Dispatch**         | Static / explicit — e.g., `visit_circle` hardcoded in `accept`.    | Dynamic — method resolved at runtime via `getattr`.            |
| **Adding New Element** | Painful — must update Visitor interface and all concrete visitors. | Easy — add `visit_NewElement` only where needed.               |


#### When to use which?

- **Java Way**: If you need compile-time safety and strict enforcement that every visitor must handle every shape.
- **Pythonic Way**: Almost always preferred in Python. It keeps your data models (`Circle`, `Square`) clean and pure, moving all the "messy" business logic into the Visitor.

# Visitor Design Pattern

explained using a complex, real-world scenario: A Corporate File System Auditor.

#### The Scenario: File System Analysis

Imagine you are building a tool to manage a huge file server. You have different types of nodes:
- **PDF File**: Has properties like `is_signed`, `page_count`.
- **Excel File**: Has properties like `rows`, `contains_macros`.
- **Directory**: Contains a list of other files/directories.

**The Complexity**: You need to run completely unrelated operations on this tree:
- **Storage Report**: Calculate total size (Recursive).
- **Security Audit**: Flag risky files (e.g., unsigned PDFs or Excels with macros).
- **JSON Export**: Dump the tree structure.

If you put all this logic inside the `File` classes, they become "God Objects" with thousands of lines of code. The Visitor pattern extracts this logic.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we use Double Dispatch.
- **The Element** (`accept`): The Directory class is responsible for guiding the visitor to its children (Traversal logic is coupled with the data).
- **The Visitor** (`visit`): Defines specific methods for every single class type.

#### THE VISITOR INTERFACE

In [10]:
from abc import ABC, abstractmethod

class Visitor(ABC):
    @abstractmethod
    def visit_pdf(self, pdf): pass

    @abstractmethod
    def visit_excel(self, excel): pass

    @abstractmethod
    def visit_directory(self, directory): pass

#### THE ELEMENTS (File System Nodes)

In [12]:
from abc import ABC, abstractmethod

class FileSystemNode(ABC):
    @abstractmethod
    def accept(self, visitor: Visitor): pass

class PdfFile(FileSystemNode):
    def __init__(self, name: str, size: int, is_signed: bool):
        self.name = name
        self.size = size
        self.is_signed = is_signed

    def accept(self, visitor: Visitor):
        # Double Dispatch
        visitor.visit_pdf(self)

class ExcelFile(FileSystemNode):
    def __init__(self, name: str, size: int, has_macros: bool):
        self.name = name
        self.size = size
        self.has_macros = has_macros

    def accept(self, visitor: Visitor):
        visitor.visit_excel(self)

class Directory(FileSystemNode):
    def __init__(self, name: str):
        self.name = name
        self.children: List[FileSystemNode] = []

    def add(self, node: FileSystemNode):
        self.children.append(node)

    def accept(self, visitor: Visitor):
        # 1. Visit the directory itself
        visitor.visit_directory(self)
        
        # 2. TRAVERSAL LOGIC IS HERE (Coupled)
        # The Directory decides that the visitor must visit children immediately.
        for child in self.children:
            child.accept(visitor)

#### CONCRETE VISITORS

In [13]:
class SizeCalculator(Visitor):
    def __init__(self):
        self.total_size = 0

    def visit_pdf(self, pdf):
        self.total_size += pdf.size

    def visit_excel(self, excel):
        self.total_size += excel.size

    def visit_directory(self, directory):
        # Directories have 0 size in this example, 
        # but we need this method to satisfy the Interface.
        pass

class SecurityAuditor(Visitor):
    def __init__(self):
        self.risks = []

    def visit_pdf(self, pdf):
        if not pdf.is_signed:
            self.risks.append(f"RISK: Unsigned PDF '{pdf.name}'")

    def visit_excel(self, excel):
        if excel.has_macros:
            self.risks.append(f"RISK: Macro-enabled Excel '{excel.name}'")

    def visit_directory(self, directory):
        print(f"Scanning Folder: {directory.name}...")

#### CLIENT CODE

In [14]:
def main():
    # Setup Tree
    root = Directory("Corporate_Root")
    finance = Directory("Finance")
    
    file1 = PdfFile("Contract.pdf", 500, is_signed=True)
    file2 = PdfFile("Draft_Memo.pdf", 100, is_signed=False) # Risk
    file3 = ExcelFile("Budget.xlsx", 2000, has_macros=True) # Risk
    
    root.add(file1)
    root.add(finance)
    finance.add(file2)
    finance.add(file3)

    # 1. Calculate Size
    size_vis = SizeCalculator()
    root.accept(size_vis)
    print(f"Total System Size: {size_vis.total_size} KB")

    # 2. Security Audit
    audit_vis = SecurityAuditor()
    root.accept(audit_vis)
    print("\n--- Security Report ---")
    for risk in audit_vis.risks:
        print(risk)

if __name__ == "__main__":
    main()

Total System Size: 2600 KB
Scanning Folder: Corporate_Root...
Scanning Folder: Finance...

--- Security Report ---
RISK: Unsigned PDF 'Draft_Memo.pdf'
RISK: Macro-enabled Excel 'Budget.xlsx'


## The Pythonic Way (Separated Logic)

In Python, we decouple the "Data" from the "Algorithm" completely.
- The `FileSystem` classes are dumb `dataclasses`. They don't know about `accept` or `visitors`.
- The Visitor handles both the operation logic and the traversal logic (recursion).
- We use `singledispatch` (functional style) or a dynamic base class (similar to how Python's built-in `ast` module works).

Here is the cleanest approach using a **Base Visitor with Dynamic Dispatch**:

#### THE DATA (Pure & Dumb)

In [17]:
from dataclasses import dataclass, field
from typing import List, Union

@dataclass
class Pdf:
    name: str
    size: int
    signed: bool

@dataclass
class Excel:
    name: str
    size: int
    macros: bool

@dataclass
class Folder:
    name: str
    children: List[Union[Pdf, Excel, 'Folder']] = field(default_factory=list)

    def add(self, *items):
        self.children.extend(items)

#### THE PYTHONIC VISITOR ENGINE

In [18]:
class NodeVisitor:
    """
    Standard Python pattern (used in AST module).
    It automatically finds 'visit_ClassName' methods.
    """
    def visit(self, node):
        method_name = 'visit_' + type(node).__name__
        # Fetch the method, default to generic_visit if missing
        visitor = getattr(self, method_name, self.generic_visit)
        return visitor(node)

    def generic_visit(self, node):
        print(f"No logic defined for {type(node).__name__}")

#### COMPLEX OPERATIONS (Decoupled)

In [19]:
class ReportGenerator(NodeVisitor):
    def __init__(self):
        self.lines = []
        self.depth = 0

    def visit_Folder(self, folder: Folder):
        indent = "  " * self.depth
        self.lines.append(f"{indent}📁 {folder.name}/")
        
        # WE CONTROL TRAVERSAL HERE (Not in the data class)
        self.depth += 1
        for child in folder.children:
            self.visit(child)
        self.depth -= 1

    def visit_Pdf(self, pdf: Pdf):
        indent = "  " * self.depth
        status = "🔒" if pdf.signed else "🔓"
        self.lines.append(f"{indent}📄 {pdf.name} ({status})")

    def visit_Excel(self, excel: Excel):
        indent = "  " * self.depth
        status = "⚠️" if excel.macros else "✅"
        self.lines.append(f"{indent}📊 {excel.name} ({status})")

class VirusScan(NodeVisitor):
    """
    Simulates a different complex operation.
    """
    def visit_Folder(self, folder):
        print(f"Scanning Folder: {folder.name}...")
        # Recursion
        for child in folder.children:
            self.visit(child)

    def visit_Excel(self, excel):
        if excel.macros:
            print(f"   [ALERT] Quarantining {excel.name} (Macros detected)")

    def visit_Pdf(self, pdf):
        pass # PDFs are safe in this scenario

#### CLIENT CODE

In [20]:
def main():
    # 1. Construct Tree
    root = Folder("Server")
    docs = Folder("Documents")
    root.add(
        Pdf("Welcome.pdf", 10, True),
        docs
    )
    docs.add(
        Pdf("Unsigned_Contract.pdf", 20, False),
        Excel("Finance_Hack.xlsx", 50, True)
    )

    # 2. Run Report Visitor
    print("--- 1. Structure Report ---")
    reporter = ReportGenerator()
    reporter.visit(root) # Trigger
    print("\n".join(reporter.lines))

    # 3. Run Virus Scan Visitor
    print("\n--- 2. Virus Scan ---")
    scanner = VirusScan()
    scanner.visit(root)

if __name__ == "__main__":
    main()

--- 1. Structure Report ---
📁 Server/
  📄 Welcome.pdf (🔒)
  📁 Documents/
    📄 Unsigned_Contract.pdf (🔓)
    📊 Finance_Hack.xlsx (⚠️)

--- 2. Virus Scan ---
Scanning Folder: Server...
Scanning Folder: Documents...
   [ALERT] Quarantining Finance_Hack.xlsx (Macros detected)


#### Why the Pythonic version fits Complex scenarios better

- **Traversal Control**: In the Java version, the `Directory` class enforced `for child in children`. What if you want a visitor that stops recursively visiting if it finds a specific file? You can't, because the loop is hardcoded in the data class. In the Pythonic version (`ReportGenerator`), the visitor owns the loop. It can decide to skip children, visit them in reverse, or stop halfway.
- **Clean Models**: The `Pdf` and `Excel` classes are pure Data Classes. They contain zero logic about reporting or scanning. This is perfect for large systems where data models are shared across many modules.